# Lecture d'offres Welcome to the Jungle

Ce notebook lit toutes les offres visibles sur la page jobs d'ILLUIN Technology, sans authentification.

Il récupère la configuration publique de la page jobs, interroge l'index Algolia exposé par le site, puis appelle le endpoint détail de chaque offre pour afficher successivement un résumé pour chaque job.

In [5]:
import json
import re
from html.parser import HTMLParser
from urllib.parse import urlencode, urlparse

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

COMPANY_JOBS_URL = "https://www.welcometothejungle.com/fr/companies-v1/illuin-tech/jobs"
# COMPANY_JOBS_URL = "https://www.welcometothejungle.com/en/companies-v1/alten-1/jobs"
# COMPANY_JOBS_URL = "https://www.welcometothejungle.com/fr/companies-v1/deloitte/jobs"
# COMPANY_JOBS_URL = "https://www.welcometothejungle.com/fr/companies-v1/capgemini/jobs"
HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "fr-FR,fr;q=0.9,en;q=0.8",
}
SESSION = requests.Session()
SESSION.headers.update(HEADERS)
SESSION.mount("https://", HTTPAdapter(max_retries=Retry(
    total=2, status_forcelist=[429, 502, 503, 504],
    allowed_methods=["GET", "POST"],
)))


def fetch_json(url: str, payload: dict | None = None, extra_headers: dict | None = None) -> dict:
    response = SESSION.request(
        "POST" if payload is not None else "GET",
        url, json=payload, headers=extra_headers, timeout=(10, 30),
    )
    response.raise_for_status()
    return response.json()


def find_jobs_search_properties(node):
    if isinstance(node, dict):
        properties = node.get("properties")
        if node.get("kind") == "jobs-search" and isinstance(properties, dict) and properties.get("algolia_api_key"):
            return properties
        for value in node.values():
            found = find_jobs_search_properties(value)
            if found is not None:
                return found
    elif isinstance(node, list):
        for item in node:
            found = find_jobs_search_properties(item)
            if found is not None:
                return found
    return None


def get_company_jobs_config(company_jobs_url: str) -> dict:
    parsed = urlparse(company_jobs_url)
    path = parsed.path.rstrip("/").replace("/companies-v1/", "/companies/")
    if not re.fullmatch(r"/[^/]+/companies/[^/]+/jobs", path):
        raise ValueError("URL attendue: une page entreprise /companies[/ -v1]/.../jobs.")
    try:
        page_payload = fetch_json(
            "https://api.welcometothejungle.com/api/v1/pages?" + urlencode({"path": path})
        )
    except requests.HTTPError as exc:
        if exc.response is not None and exc.response.status_code == 404:
            raise ValueError(
                f"Page jobs introuvable (404): {company_jobs_url}. "
                f"Chemin API: {path}. "
                "Vérifie l'adresse publique de l'entreprise; la langue de l'URL est conservée."
            ) from exc
        raise
    jobs_search = find_jobs_search_properties(page_payload)
    if not jobs_search:
        raise ValueError(f"Configuration jobs-search absente pour {company_jobs_url}.")
    return {
        "company_slug": jobs_search["website_organization"]["slug"],
        "filters": jobs_search["filters"],
        "default_job": jobs_search.get("default_job"),
        "algolia_api_key": jobs_search["algolia_api_key"],
        "algolia_index_name": jobs_search["algolia_index_name"],
        "algolia_app_id": jobs_search["algolia_app_id"],
        "company_jobs_url": company_jobs_url,
        "origin": f"{parsed.scheme}://{parsed.netloc}",
    }


def build_algolia_filters(filters: dict) -> str:
    parts = []
    for key, values in filters.items():
        values = values if isinstance(values, list) else [values]
        alternatives = [f"{key}:{json.dumps(value)}" for value in values]
        if alternatives:
            parts.append("(" + " OR ".join(alternatives) + ")")
    return " AND ".join(parts)


def fetch_job_hits(config: dict) -> list[dict]:
    app_id = config["algolia_app_id"]
    endpoint = f"https://{app_id.lower()}-dsn.algolia.net/1/indexes/*/queries"
    headers = {
        "X-Algolia-Application-Id": app_id,
        "X-Algolia-API-Key": config["algolia_api_key"],
        "Origin": config["origin"],
        "Referer": config["company_jobs_url"],
    }
    hits = []
    page = 0
    while True:
        payload = {"requests": [{
            "indexName": config["algolia_index_name"],
            "params": urlencode({
                "query": "", "filters": build_algolia_filters(config["filters"]),
                "hitsPerPage": 100, "page": page,
            }),
        }]}
        result = fetch_json(endpoint, payload, headers)["results"][0]
        if "hits" not in result:
            raise ValueError(f"Recherche jobs impossible: {result.get('message', result)}")
        hits.extend(result["hits"])
        page += 1
        if page >= result["nbPages"]:
            break
    default_job = config.get("default_job") or {}
    if default_job.get("slug"):
        hits.append(default_job)
    unique_hits = {}
    for hit in hits:
        if hit.get("slug"):
            unique_hits.setdefault(hit["slug"], hit)
    return list(unique_hits.values())


def fetch_job_detail(company_slug: str, job_slug: str) -> dict:
    return fetch_json(
        f"https://api.welcometothejungle.com/api/v1/organizations/{company_slug}/jobs/{job_slug}"
    )["job"]


class TextExtractor(HTMLParser):
    def __init__(self) -> None:
        super().__init__()
        self.parts = []

    def handle_data(self, data: str) -> None:
        if data.strip():
            self.parts.append(data.strip())

    def get_text(self) -> str:
        return " ".join(self.parts)


def html_to_text(raw_html) -> str:
    if not raw_html:
        return ""
    cleaned = re.sub(r"<(script|style)[^>]*>.*?</\1>", " ", raw_html, flags=re.IGNORECASE | re.DOTALL)
    parser = TextExtractor()
    parser.feed(cleaned)
    return re.sub(r"\s+", " ", parser.get_text()).strip()


def summarize_offer_from_job(job: dict) -> dict:
    organization = job.get("organization") or {}
    office = job.get("office") or {}
    contract_type = job.get("contract_type")
    return {
        "title": job.get("name"),
        "company": organization.get("name"),
        "employment_type": contract_type.upper() if isinstance(contract_type, str) else None,
        "location": office.get("address"),
        "date_posted": job.get("published_at"),
        "valid_through": None,
        "company_website": organization.get("media_website_url"),
        "description_preview": html_to_text(job.get("description"))[:1200],
    }


def print_offer_summary(summary: dict, index: int, total: int) -> None:
    print(f"=== Offre {index}/{total} ===")
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print("\nAperçu de la description:")
    print(summary["description_preview"])
    print()

In [6]:
jobs_config = get_company_jobs_config(COMPANY_JOBS_URL)
job_hits = fetch_job_hits(jobs_config)
summaries = []
errors = []
total = len(job_hits)
print(f"Entreprise: {jobs_config['company_slug']} | {total} offres trouvées", flush=True)

for index, hit in enumerate(job_hits, start=1):
    try:
        company_slug = (hit.get("organization") or {}).get("slug") or jobs_config["company_slug"]
        job = fetch_job_detail(company_slug, hit["slug"])
        summary = summarize_offer_from_job(job)
    except (requests.RequestException, ValueError, KeyError) as exc:
        errors.append({"slug": hit.get("slug"), "error": str(exc)})
        print(f"=== Offre {index}/{total} ===")
        print(f"Impossible de lire l'offre: {hit.get('slug')}: {exc}", flush=True)
        continue

    summaries.append(summary)
    print_offer_summary(summary, index, total)

print(f"Terminé: {len(summaries)}/{total} offres lues, {len(errors)} erreurs.", flush=True)

Entreprise: illuin-tech | 14 offres trouvées
=== Offre 1/14 ===
{
  "title": "Lead Delivery /Senior Project Manager (F/H)  ",
  "company": "ILLUIN Technology",
  "employment_type": "FULL_TIME",
  "location": "Puteaux, Ile-de-France, France",
  "date_posted": "2026-10-26T00:00:40Z",
  "valid_through": null,
  "company_website": "https://www.illuin.tech/",
  "description_preview": "LE CONTEXTE : Fondée en 2017 par des ingénieurs de CentraleSupélec, ILLUIN Technology est une scale-up française en pleine effervescence de plus de 200 passionnés. Notre mission ? Déployer l'IA et les solutions digitales de pointe à grande échelle, avec une culture de l’excellence forgée sur le plateau de Paris-Saclay. Notre force repose sur une double approche unique : MAKE : Des réalisations tech sur-mesure en GenAI, Data Science et Software Engineering. BUY : Une suite de produits propriétaires de pointe ( Dialogue, Automate, Analyze ) orchestrée par notre plateforme low-code nAixt . Propulsés par une crois